# AEGEAN | Flight Review Scraper

Collect review links, ratings and review text; preserve incremental CSV and JSON outputs.

**Dependencies:** requests beautifulsoup4 pandas. Install in the active kernel before running.

**Execution:** Run the setup, configuration and collection cells in order. Review airline selections and limits before collection. Existing data must be migrated to `Data/Scraping/flight_report_data/` before resuming.

**Outputs:** All generated files are stored in `Data/Scraping/flight_report_data/`. Existing filenames and schemas are retained.

Collection requires network access; browser-based scrapers also require Chrome.

## 1. Portable project paths

In [ ]:
from pathlib import Path

# Locate the project from the notebook kernel's working directory.
# Start Jupyter inside the project tree; the root name is not significant.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if all((p / name).is_dir() for name in ("Python Code", "Data", "Streamlit"))),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project folder containing Python Code, Data and Streamlit.")

SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
SOURCE_DIR = SCRAPING_DIR / 'flight_report_data'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)


## Imports and prerequisites

In [ ]:
import re
import json
import time
import random
import requests
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs

## Configuration

In [ ]:
AIRLINE_URLS = [
    "https://flight-report.com/en/airline/96/aegean-airlines/?page=1",
    "https://flight-report.com/en/airline/3319/lufthansa/?page=1",
    "https://flight-report.com/en/airline/137/air-france/?page=1",
    "https://flight-report.com/en/airline/4946/turkish-airlines/?page=1",
    "https://flight-report.com/en/airline/1421/bangkok-airways/?page=1",
    "https://flight-report.com/en/airline/4020/porter-airlines/",
    "https://flight-report.com/en/airline/5350/virgin-australia/"

    # Add more airline listing URLs here
]

BASE_URL = "https://flight-report.com"

OUTPUT_DIR = SOURCE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_PAGES = None       # None = all pages
MAX_REVIEWS = None    # None = all reviews per airline
DELAY = (1.5, 3.0)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; ResearchScraper/1.0)",
    "Accept-Language": "en-US,en;q=0.9"
})

## Helpers

In [ ]:
def clean(text):
    """Normalize whitespace in text."""
    return re.sub(r"\s+", " ", text or "").strip()


def get_soup(url):
    """Fetch a page and parse its HTML."""
    response = session.get(url, timeout=30)
    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")


def sleep():
    """Apply the configured randomized request delay."""
    time.sleep(random.uniform(*DELAY))


def get_airline_name(url):
    """Derive the airline slug from its listing URL."""
    path = urlparse(url).path.rstrip("/")
    slug = path.split("/")[-1]
    return slug.replace("-", "_")


def save_records(records, csv_path, json_path):
    """Persist review records in CSV and JSON formats."""
    df = pd.DataFrame(records)

    if "review_id" in df.columns:
        df = df.drop_duplicates(
            subset=["review_id"],
            keep="last"
        )

    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    df.to_json(
        json_path,
        orient="records",
        indent=2,
        force_ascii=False
    )

    return df

## Step 1: Collect Review Links

In [ ]:
def get_review_links(airline_url):
    """Discover individual review URLs from listing pages."""
    links = {}

    page = 1

    while True:

        if MAX_PAGES is not None and page > MAX_PAGES:
            break

        url = airline_url.split("?")[0] + f"?page={page}"

        print(f"  Listing page {page}: {url}")

        try:
            soup = get_soup(url)

        except requests.RequestException as e:
            print(f"  Listing request failed: {e}")
            break

        new_links = 0

        for a in soup.select('a[href*="/en/report/"]'):

            full_url = urljoin(
                BASE_URL,
                a.get("href", "")
            )

            match = re.search(
                r"/en/report/(\d+)/",
                full_url
            )

            if not match:
                continue

            review_id = match.group(1)

            if review_id not in links:
                links[review_id] = full_url
                new_links += 1

            if (
                MAX_REVIEWS is not None
                and len(links) >= MAX_REVIEWS
            ):
                break

        print(
            f"  New links: {new_links} | "
            f"Total: {len(links)}"
        )

        if new_links == 0:
            break

        if (
            MAX_REVIEWS is not None
            and len(links) >= MAX_REVIEWS
        ):
            break

        page += 1
        sleep()

    return list(links.values())

## Step 2: Extract Airline Ratings

In [ ]:
def extract_rating_section(soup):

    """Parse airline rating categories from the review page."""
    ratings = {
        "airline_overall": None,
        "airline_cabin": None,
        "airline_cabin_crew": None,
        "airline_entertainment_wifi": None,
        "airline_meal_catering": None
    }

    lines = [
        clean(line)
        for line in soup.get_text(
            "\n", strip=True
        ).splitlines()
    ]

    lines = [line for line in lines if line]

    try:
        start = next(
            i for i, line in enumerate(lines)
            if line.lower() == "product ratings"
        )

        airline_start = next(
            i for i in range(start + 1, len(lines))
            if lines[i].lower() == "airline"
        )

    except StopIteration:
        return ratings

    end = len(lines)

    for i in range(airline_start + 1, len(lines)):

        if lines[i].lower() in [
            "lounge",
            "departure airport",
            "arrival airport",
            "arrival airport ratings",
            "conclusion"
        ]:
            end = i
            break

    section = "\n".join(
        lines[airline_start + 1:end]
    )

    patterns = {
        "airline_cabin": r"\bCabin\b",
        "airline_cabin_crew": r"\bCabin crew\b",
        "airline_entertainment_wifi":
            r"\bEntertainment\s*/\s*wifi\b",
        "airline_meal_catering":
            r"\bMeal\s*/\s*catering\b"
    }

    number = r"(\d+(?:\.\d+)?)\s*(?:/\s*10)?"

    for key, label_pattern in patterns.items():

        match = re.search(
            label_pattern + r"\s*" + number,
            section,
            flags=re.IGNORECASE
        )

        if match:
            value = float(match.group(1))

            if 0 <= value <= 10:
                ratings[key] = value

    component_keys = [
        "airline_cabin",
        "airline_cabin_crew",
        "airline_entertainment_wifi",
        "airline_meal_catering"
    ]

    values = [
        ratings[key]
        for key in component_keys
        if ratings[key] is not None
    ]

    if len(values) == 4:
        ratings["airline_overall"] = round(
            sum(values) / 4, 2
        )

    return ratings

## Step 3: Extract Review Text

In [ ]:
def extract_review_text(soup):

    """Extract textual review sections from the page."""
    start = soup.find(
        string=lambda s: s and re.search(
            r"Flight routing|INTRO|Introduction",
            s,
            re.I
        )
    )

    end = soup.find(
        string=lambda s: s and
        "Product ratings" in s
    )

    if not start or not end:
        return None

    paragraphs = []

    for element in start.parent.next_elements:

        if element is end.parent:
            break

        if getattr(element, "name", None) == "p":

            value = clean(
                element.get_text(" ", strip=True)
            )

            if value and value not in paragraphs:
                paragraphs.append(value)

    return "\n".join(paragraphs) or None

## Step 4: Scrape Individual Review

In [ ]:
def scrape_review(url, airline_name):

    """Collect the fields for one individual review."""
    soup = get_soup(url)

    match = re.search(
        r"/en/report/(\d+)/",
        url
    )

    review = {
        "source": "Flight-Report",
        "airline": airline_name,
        "url": url,
        "review_id": match.group(1) if match else None
    }

    h1 = soup.find("h1")

    review["title"] = (
        clean(h1.get_text(" ", strip=True))
        if h1 else None
    )

    text = soup.get_text("\n", strip=True)

    # Flight date
    flight_date = re.search(
        r"Flight taken on\s+([^\n]+)",
        text
    )

    review["flight_date_raw"] = (
        clean(flight_date.group(1))
        if flight_date else None
    )

    # Publication date
    publication_date = re.search(
        r"\b(20\d{2}-\d{2}-\d{2})\s+"
        r"\d{2}:\d{2}:\d{2}",
        text
    )

    review["publication_date"] = (
        publication_date.group(1)
        if publication_date else None
    )

    # Author
    author = soup.find(
        string=lambda s: s and
        "Flight review by" in s
    )

    review["author"] = None

    if author:
        heading = author.find_parent(
            ["h2", "h3", "h4"]
        )

        if heading:
            review["author"] = clean(
                heading.get_text(" ", strip=True)
            ).replace(
                "Flight review by", ""
            ).strip()

    # Flight number
    flight_number = re.search(
        r"\b[A-Z0-9]{2}\d{2,4}\b",
        text
    )

    review["flight_number"] = (
        flight_number.group(0)
        if flight_number else None
    )

    # Full review text
    review["review_text"] = extract_review_text(soup)

    # Conclusion
    conclusion = soup.find(
        string=lambda s: s and
        clean(s) == "Conclusion"
    )

    review["conclusion"] = None

    if conclusion:

        heading = conclusion.find_parent(
            ["h2", "h3", "h4"]
        )

        if heading:

            parts = []

            for sibling in heading.next_siblings:

                if getattr(
                    sibling, "name", None
                ) in ["h2", "h3"]:
                    break

                if getattr(
                    sibling, "name", None
                ):
                    parts.append(
                        clean(
                            sibling.get_text(
                                " ", strip=True
                            )
                        )
                    )

            review["conclusion"] = (
                " ".join(parts) or None
            )

    # Airline ratings
    review.update(
        extract_rating_section(soup)
    )

    return review

## Step 5: Process Each Airline

In [ ]:
def process_airline(airline_url):

    """Process one airline and persist its incremental results."""
    airline_name = get_airline_name(
        airline_url
    )

    print(f"\n{'=' * 60}")
    print(f"AIRLINE: {airline_name}")
    print(f"{'=' * 60}")

    airline_dir = OUTPUT_DIR / airline_name
    airline_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    csv_path = (
        airline_dir / f"{airline_name}_reviews.csv"
    )

    json_path = (
        airline_dir / f"{airline_name}_reviews.json"
    )

    # Load previously collected reviews
    existing = {}

    if json_path.exists():

        try:
            with open(
                json_path,
                "r",
                encoding="utf-8"
            ) as f:
                previous = json.load(f)

            existing = {
                str(r["review_id"]): r
                for r in previous
                if r.get("review_id") is not None
            }

            print(
                f"Previously saved: "
                f"{len(existing)} reviews"
            )

        except (OSError, ValueError) as e:
            print(f"Could not load existing JSON: {e}")

    # Collect links
    links = get_review_links(
        airline_url
    )

    print(f"Total review links: {len(links)}")

    # Scrape only missing reviews
    for i, url in enumerate(links, start=1):

        match = re.search(
            r"/en/report/(\d+)/",
            url
        )

        if not match:
            continue

        review_id = match.group(1)

        if review_id in existing:
            continue

        print(
            f"[{i}/{len(links)}] "
            f"Scraping review {review_id}"
        )

        try:
            review = scrape_review(
                url,
                airline_name
            )

            existing[review_id] = review

            # Save progress after each review
            save_records(
                list(existing.values()),
                csv_path,
                json_path
            )

        except requests.RequestException as e:
            print(f"Request error: {e}")

        except Exception as e:
            print(f"Parsing error: {e}")

        sleep()

    df_airline = save_records(
        list(existing.values()),
        csv_path,
        json_path
    )

    print(
        f"Completed {airline_name}: "
        f"{len(df_airline)} reviews"
    )

    return df_airline

## Step 6: Run All Airlines

In [ ]:
all_data = []

for airline_url in AIRLINE_URLS:

    try:
        df_airline = process_airline(
            airline_url
        )

        if not df_airline.empty:
            all_data.append(df_airline)

    except Exception as e:
        print(
            f"Airline failed: "
            f"{airline_url} | {e}"
        )

## Step 7: Combined Output

In [ ]:
# Rebuild combined output from all saved airline folders,
# including airlines processed in earlier runs.

saved_frames = []

for path in OUTPUT_DIR.glob("*/*_reviews.json"):

    try:
        saved = pd.read_json(path)

        if not saved.empty:
            saved_frames.append(saved)

    except (ValueError, OSError) as e:
        print(f"Could not read {path}: {e}")

if saved_frames:

    df_all = pd.concat(
        saved_frames,
        ignore_index=True
    )

    df_all = df_all.drop_duplicates(
        subset=["review_id"],
        keep="last"
    )

    df_all.to_csv(
        OUTPUT_DIR / "all_airlines_reviews.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df_all.to_json(
        OUTPUT_DIR / "all_airlines_reviews.json",
        orient="records",
        indent=2,
        force_ascii=False
    )

    print("\nALL AIRLINES COMPLETED")
    print(f"Total reviews: {len(df_all)}")

    display(
        df_all.groupby("airline")
        .agg(
            reviews=("review_id", "count"),
            average_rating=("airline_overall", "mean")
        )
        .round(2)
    )

else:
    print("No reviews collected.")

## Inspect saved data / checkpoint status

In [ ]:
df_all.tail(10)

## Inspect saved data / checkpoint status

In [ ]:
import pandas as pd

# Read the combined CSV containing all airline reviews
df = pd.read_csv(
    SOURCE_DIR / "all_airlines_reviews.csv",
    encoding="utf-8-sig"
)

print(f"Total reviews: {len(df)}")
print(f"Total airlines: {df['airline'].nunique()}")

display(df.head(10))